# Instacart Lakehouse Analytics — Bronze Ingestion

## Objective

This notebook ingests the prepared Instacart CSV files from a Databricks Unity Catalog volume and stores them as managed Bronze Delta tables.

The Bronze layer preserves the source data with minimal transformation while adding ingestion metadata for traceability.

## Inputs

- Orders
- Prior order-product transactions
- Train order-product transactions
- Products
- Aisles
- Departments

## Outputs

Six managed Delta tables are created in `workspace.default` with the prefix `bronze_`.

## 1. Environment Setup

Import the required PySpark functions and define the Unity Catalog volume path containing the raw project files.

In [0]:
from pyspark.sql import functions as F

base_path = "/Volumes/workspace/default/instacart_raw"

## 2. Source File Configuration

Map each logical dataset name to its corresponding CSV file in the managed volume. Centralizing the paths makes the ingestion process easier to maintain and reuse.

In [0]:
files = {
    "orders": f"{base_path}/orders_project.csv",
    "products": f"{base_path}/products.csv",
    "aisles": f"{base_path}/aisles.csv",
    "departments": f"{base_path}/departments.csv",
    "order_products_prior": f"{base_path}/order_products_prior_project.csv",
    "order_products_train": f"{base_path}/order_products_train_project.csv"
}

## 3. Load Raw CSV Files

Read each source file into a Spark DataFrame using headers and inferred schemas. Row counts and schemas are reviewed before the data is written to Delta tables.

In [0]:
dataframes = {}

for name, path in files.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )

    dataframes[name] = df

    print(f"{name}: {df.count():,} rows")
    df.printSchema()

orders: 109,467 rows
root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)

products: 49,688 rows
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)

aisles: 134 rows
root
 |-- aisle_id: integer (nullable = true)
 |-- aisle: string (nullable = true)

departments: 21 rows
root
 |-- department_id: integer (nullable = true)
 |-- department: string (nullable = true)

order_products_prior: 750,000 rows
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)

order_products_train: 380,61

In [0]:
display(dataframes["orders"].limit(10))

order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
23391,7,prior,17,0,10,28.0
525192,7,train,21,2,11,6.0
68288,10,prior,2,5,15,30.0
1822501,10,train,6,0,19,30.0
19256,13,prior,4,1,12,9.0
1827621,13,train,13,0,21,8.0
62373,21,prior,5,1,14,7.0
77791,21,prior,19,3,9,8.0
1854765,21,train,34,1,12,28.0
8382,23,prior,2,0,10,9.0


In [0]:
display(dataframes["order_products_prior"].limit(10))

order_id,product_id,add_to_cart_order,reordered
2,33120,1,1
2,28985,2,1
2,9327,3,0
2,45918,4,1
2,30035,5,0
2,17794,6,1
2,40141,7,1
2,1819,8,1
2,43668,9,0
3,33754,1,1


## 4. Correct Product File Parsing

Some product names contain embedded commas and quotation marks. The default CSV inference produced malformed numeric fields for these records.

The product file is therefore parsed using Python's CSV reader and an explicit Spark schema to preserve product names and ensure numeric identifiers remain valid.

In [0]:
import csv
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    LongType,
    StringType
)

products_path = (
    "/Volumes/workspace/default/"
    "instacart_raw/products.csv"
)

product_rows = []

with open(products_path, mode="r", encoding="utf-8-sig", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        product_rows.append(
            (
                int(row["product_id"]),
                row["product_name"],
                int(row["aisle_id"]),
                int(row["department_id"])
            )
        )

product_schema = StructType([
    StructField("product_id", LongType(), False),
    StructField("product_name", StringType(), True),
    StructField("aisle_id", LongType(), True),
    StructField("department_id", LongType(), True)
])

products_fixed = spark.createDataFrame(
    product_rows,
    schema=product_schema
)

products_fixed = (
    products_fixed
    .withColumn("source_file", F.lit(products_path))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

products_fixed.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.bronze_products")

print("bronze_products corrected:", products_fixed.count())

bronze_products corrected: 49688


In [0]:
display(
    spark.table("workspace.default.bronze_products")
    .filter(F.col("product_name").contains("Blunted"))
)

product_id,product_name,aisle_id,department_id,source_file,ingestion_timestamp
6816,"Scotch Kids 5\"" Scissors, Blunted, Red",87,17,/Volumes/workspace/default/instacart_raw/products.csv,2026-06-23T19:49:43.141Z


## 5. Add Ingestion Metadata

Add the source file path and ingestion timestamp to each dataset. These fields support lineage, traceability, and troubleshooting.

In [0]:
from pyspark.sql import functions as F

bronze_dataframes = {}

for name, df in dataframes.items():
    bronze_df = (
        df
        .withColumn("source_file", F.lit(files[name]))
        .withColumn("ingestion_timestamp", F.current_timestamp())
    )

    bronze_dataframes[name] = bronze_df

## 6. Write Bronze Delta Tables

Store each DataFrame as a managed Delta table in Unity Catalog. Overwrite mode is used because this portfolio pipeline performs a full refresh during development.

In [0]:
# Full-refresh overwrite is appropriate for the current portfolio implementation
table_names = {
    "orders": "bronze_orders",
    "products": "bronze_products",
    "aisles": "bronze_aisles",
    "departments": "bronze_departments",
    "order_products_prior": "bronze_order_products_prior",
    "order_products_train": "bronze_order_products_train"
}

for source_name, table_name in table_names.items():
    full_table_name = f"workspace.default.{table_name}"

    (
        bronze_dataframes[source_name]
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(full_table_name)
    )

    print(f"Created: {full_table_name}")

Created: workspace.default.bronze_orders


---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5357197377700863>, line 19
     11 for source_name, table_name in table_names.items():
     12     full_table_name = f"workspace.default.{table_name}"
     14     (
     15         bronze_dataframes[source_name]
     16         .write
     17         .format("delta")
     18         .mode("overwrite")
---> 19         .saveAsTable(full_table_name)
     20     )
     22     print(f"Created: {full_table_name}")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 sel

## 7. Validate Bronze Outputs

Confirm that every Bronze table was created successfully and that its row count matches the corresponding uploaded source file.

In [0]:
for table_name in table_names.values():
    full_table_name = f"workspace.default.{table_name}"
    row_count = spark.table(full_table_name).count()
    print(f"{full_table_name}: {row_count:,} rows")

In [0]:
display(
    spark.table("workspace.default.bronze_orders")
    .limit(10)
)

## Bronze Layer Summary

The six Instacart source datasets were successfully ingested and stored as managed Delta tables.

### Tables created

- `bronze_orders`
- `bronze_order_products_prior`
- `bronze_order_products_train`
- `bronze_products`
- `bronze_aisles`
- `bronze_departments`

The next notebook cleans, standardizes, and enriches these datasets to produce the Silver layer.